<hr style="border:4px solid black">

<p style="text-align: center; font-size: x-large; font-weight: bold;"> Perform Portfolio Analysis - Summary Stats </p>
<br>
<p style="text-align: center; font-size: large;font-weight: bold;"> For MO Quant Strategies Backtest </p>

<hr style="border:4px solid black">

<a id="Index"></a>
<h1> Index </h1>

---

---

- [Section - 1](#S1): Description
- [Section - 2](#S2): Import Libraries
- [Section - 3](#S3): Functions - Portfolio Performance
- [Section - 4](#S4): Functions - Rolling Performance
- [Section - 5](#S5): Functions - Calender Year Performance
- [Section - 6](#S6): Functions - Point in Time - Trailing Returns
- [Section - 7](#S7): Run Code and Save on Excel
<br>

---

---

<hr style="border:4px solid black">

[Go To Index](#Index)
<a id="S1"></a>

# Description

- Import file as daily/monthly/yearly returns with dates
- Run Performance Stats:
    - Periodic Returns with choice of sub-periods @ Function - Portfolio Performance
    - Rolling Returns with choice of running it as daily/monthly/yearly rolling @ Function - Rolling Performance
    - Calender Year Returns @ Function - Calender Year Performance
    - Trailing Returns @ Function - Point in Time - Trailing Returns
- The fuctions can handle daily/monthly/yearly returns. The data timeline has to be punched in as function input
- The function can handle multiple return series as long as they have the same timeline

<hr style="border:4px solid black">

[Go To Index](#Index)
<a id="S2"></a>

# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import os
import matplotlib.pyplot as plt
import io

from matplotlib.ticker import StrMethodFormatter
import seaborn as sns

<hr style="border:4px solid black">

[Go To Index](#Index)
<a id="S3"></a>

# Functions

---

## Portfolio Performance

In [2]:
def check_start_end_year_p(data_df, start_year, end_year):
    
    # Bring Data DF in correct Format
    data = data_df.copy(deep = True)
    date_col = data.index.name
    data = data.reset_index()
    
    # Extract Years
    if isinstance(data[date_col].dtype, pd.core.dtypes.dtypes.PeriodDtype):
        data[date_col] = data[date_col].dt.to_timestamp().dt.year
    else:
        data[date_col] = pd.to_datetime(data[date_col], errors='coerce').dt.year
    
    year_list = data[date_col].unique().tolist()
    
    if (start_year not in year_list) or (end_year not in year_list):
        raise ValueError("Quoted start or end year not in the dataframe")
    else:
        return True

In [3]:
def trim_df_year_p(data_df, start_year, end_year):

        # Integer to string
        start_year = str(start_year)
        end_year = str(end_year)
    
        # Fix Dates if not in correct format
        date_col = data_df.index.name
        data_df = data_df.reset_index()
    
        if isinstance(data_df[date_col].dtype, pd.core.dtypes.dtypes.PeriodDtype):
            data_df[date_col] = pd.to_datetime(data_df[date_col].dt.to_timestamp())
        else:
            data_df[date_col] = pd.to_datetime(data_df[date_col], errors='coerce')

        # Reset Index
        data_df = data_df.set_index(date_col)
        
        # Trim Dataframe
        data_df = data_df.loc[start_year:end_year, :]

        #Voila/Hari Om
        return data_df

In [4]:
def max_drawdown_p(data_df):
    log_drawdowns = (np.log(1 + data_df).cumsum()) - (np.log(1 + data_df).cumsum().cummax()) 
    max_drawdown = (np.exp(log_drawdowns) - 1).min()
    return np.round(max_drawdown, 4)

In [5]:
def downside_dev_p(data_series, annual_factor = 252):
    return (data_series.loc[data_series < 0].std() * np.sqrt(annual_factor))

In [6]:
def cumulative_returns_p(data_df):
    c_r = data_df.add(1).cumprod().sub(1)
    c_r = c_r.values[-1]
    return c_r

In [7]:
def annualized_returns_p(data_array, annual_factor, no_data_points):
    annualized_formula = annual_factor / no_data_points
    a_r = (1 + data_array) ** annualized_formula
    a_r = a_r - 1
    return a_r

In [8]:
def annualized_sd_p(data_df, annual_factor):
    annualized_formula = np.sqrt(annual_factor)
    a_risk = data_df.std()
    a_risk = a_risk * annualized_formula
    a_risk = a_risk.values
    return a_risk

In [9]:
def return_risk_ratio_p(return_array, risk_array):
    return return_array / risk_array

In [10]:
def compute_portfolio_performance(data_df, data_df_timeline = "daily", start_year = None, end_year = None):
    """
    -data: Quote a dataframe
    -data_timeline: daily or monthly or yearly
    -start_year: Trim DF to start from start_year; Default: None
    -end_year: Trim DF to end at end_year; Default: None
    """
    # We assume there is no start and end year quoted
    check_years = False 
    
    # Figure out factor to annualize data
    data_df_timeline = data_df_timeline.strip().lower()
    if data_df_timeline not in ["daily", "monthly", "yearly"]:
        raise ValueError("data_timeline is wrongly quoted")

    periods_factor_dict = {"daily": 252, "monthly": 12, "yearly": 1}
    
    annual_factor = periods_factor_dict[data_df_timeline]

    # Check Start and End Year in DF -if quoted
    if start_year is None and end_year is not None:
        raise ValueError("Quoted end_year but not start_year")
    elif start_year is not None and end_year is None:
        raise ValueError("Quoted start_year but not end_year")
        
    elif start_year is not None and end_year is not None:
        if not isinstance(start_year, int):
            raise ValueError("start_year should be an integer") 
        if not isinstance(end_year, int):
            raise ValueError("end_year should be an integer") 
        check_years = check_start_end_year_p(data_df, start_year, end_year)

    # If Start and End year trim dataframe
    if check_years:
        data_df = trim_df_year_p(data_df, start_year, end_year)

    # Cumulative Returns
    cum_ret = cumulative_returns_p(data_df)
    
    # Annual Returns
    ann_ret = annualized_returns_p(cum_ret, annual_factor, data_df.shape[0]) 

    # Annual Risk
    ann_risk = annualized_sd_p(data_df, annual_factor)

    # Annual Sharpe
    ann_sharpe = return_risk_ratio_p(ann_ret, ann_risk)

    # Annual Downside Risk
    downside_risk = data_df.apply(lambda x: downside_dev_p(x, annual_factor)).values

    # Annual Sortino Risk
    ann_sortino = return_risk_ratio_p(ann_ret, downside_risk)

    # Cumulative Drawdown
    period_drawdown = max_drawdown_p(data_df)

    # Skewness and Kurtosis
    skewness = data_df.skew().values
    kurtosis = data_df.kurt().values

    # Up and Down Days/Months
    no_of_up_periods = (data_df > 0).sum().values / data_df.shape[0]
    no_of_down_periods = 1 - no_of_up_periods
    
    # Put Together all Metrics and build DF
    cum_ret_scaled = (cum_ret * 100) + 100 # Add Initial investmnet @ Growth of Rs 100
    ann_ret_scaled = ann_ret * 100
    ann_risk_scaled = ann_risk * 100
    period_drawdown_scaled = period_drawdown * 100
    no_of_up_periods_scaled = no_of_up_periods * 100
    no_of_down_periods_scaled = no_of_down_periods * 100
    
    strategy_returns = ([cum_ret_scaled, ann_ret_scaled, ann_risk_scaled, ann_sharpe, 
                         period_drawdown_scaled, ann_sortino, skewness, kurtosis, 
                         no_of_up_periods_scaled, no_of_down_periods_scaled]
                       )
    
    strategy_returns_df = (pd.DataFrame(strategy_returns,
                            index = ["G-Rs100", "AReturns", "ARisk", "Sharpe", "DDown", "Sortino", 
                                     "Skewness", "Kurtosis", "%Up_Periods", "%Down_Periods"],
                                 columns = data_df.columns)
                          )
                          

    # Round it up
    strategy_returns_df = np.round(strategy_returns_df, 2)

    # Voila/Hari Om
    return strategy_returns_df.transpose()

In [11]:
# Blank for test

In [12]:
# Blank for test

---

[Go To Index](#Index)
<a id="S4"></a>

## Rolling Performance

In [13]:
def _check_input_roll_performance(data_df_timeline, roll_period, roll_type):
    """
    -data_df_timeline: The timeline of the dataframe quoted - daily or monthly or yearly
    -roll_period: 
                -Daily-Rolling @ 1 to 200 days
                -Monthly-Rolling @ 1 to 11 months
                -Yearly-Rolling @ 1 to 10 years
    -roll_type: Daily Rolling or Monthly Rolling or Yearly Rolling 
    """
    
    # Check for Quoted consistencies
    ## Fix Input Case
    data_df_timeline = data_df_timeline.strip().lower()
    roll_type = roll_type.strip().lower()

    ## Check if data_df_timeline or roll_type in [daily, monthly, yearly]
    if data_df_timeline not in ["daily", "monthly", "yearly"]:
        raise ValueError("data_df_timeline is wrongly quoted")
    if roll_type not in ["daily", "monthly", "yearly"]:
        raise ValueError("roll_type is wrongly quoted")

    ## Check if data_df_timeline is Yearly, cant run rolling daily or monthly
    if (data_df_timeline == "yearly") & ((roll_type == "monthly") | (roll_type == "daily")):
        raise ValueError("data_df_timeline for Yearly, cant be run for roll_type Monthly or Daily")
    ## Check if data_df_timeline is Monthly, cant run rolling daily
    if (data_df_timeline == "monthly") & (roll_type == "daily"):
        raise ValueError("data_df_timeline for Monthly, cant be run for roll_type Daily")

    ## Check limit of roll_period for each roll_type
    ### If Daily is roll_type, you can run at most 1-to-200-days rolling
    if roll_type == "daily":
        if not roll_period in range(1, 201):
          raise ValueError("for roll_type Daily, roll_period limit from 1 to 200 days")
    ### If Monthly is roll_type, you can run at most 1-to-11-months rolling
    elif roll_type == "monthly":
        if not roll_period in range(1, 12):
          raise ValueError("for roll_type Monthly, roll_period limit from 1 to 11 months")
    ### If Yearly is roll_type, you can run at most 1-to-10-years rolling
    else:
        if not roll_period in range(1, 11): 
            raise ValueError("for roll_type Yearly, roll_period limit from 1 to 10 years")
    
    return None

In [14]:
def _scale_factor_roll_performance(data_df_timeline, roll_type):
    # Rolling factor to scale data_df_timeline data to roll_type
    ## eg: If we have daily data, and you want to roll it as yearly, we need to scale it bt 252-days

    ## Dictionary
    _dict_roll_type_scaling_factor = ({"daily_daily": 1, "daily_monthly": 21, "daily_yearly": 252,
                               "monthly_monthly": 1, "monthly_yearly": 12,
                               "yearly_yearly":1,}
                              )

    ## Build User Input
    _user_input_roll_type = data_df_timeline + "_" + roll_type

    ## Return Scale Factor
    return _dict_roll_type_scaling_factor[_user_input_roll_type]

In [15]:
def _annualization_factor_roll_performance(data_df_timeline):
    # Fetch factor to annualzie data
    ## eg: If we have daily data we annualize it by using 252 days
    _roll_period_annual_dict = {"daily": 252, "monthly": 12, "yearly": 1}
    return _roll_period_annual_dict[data_df_timeline]

In [16]:
def _compute_probability_roll_performance(data_df, lower_bound = None, upper_bound = None):
    """
    -Compute Probability of Returns in different buckets
    -lower_bound min: -0.20
    -upper_bound max: +0.50
    """
    # Check Input
    if lower_bound is not None and upper_bound is not None and upper_bound < lower_bound:
        raise ValueError("upper_bound can't be less than lower_bound")
    elif lower_bound is not None and lower_bound < -0.20:
        raise ValueError("lower_bound can't be less than -0.20")
    elif upper_bound is not None and upper_bound > 0.50:
        raise ValueError("upper_bound can't be more than +0.50")

    # Run code
    if lower_bound is None:
        probability = (data_df < upper_bound).mean()
    elif upper_bound is None:
        probability = (data_df > lower_bound).mean()
    else:
        probability = ((data_df > lower_bound) & (data_df < upper_bound)).mean()

    # Voila/Hari-Om
    return probability

In [17]:
def compute_rolling_performance(data_df, data_df_timeline = "daily", roll_period = 1, roll_type = "yearly"):
    """
    -data_df: Quote a dataframe
    -data_df_timeline: The timeline of the dataframe quoted - daily or monthly or yearly
    -roll_period: 
                -Daily-Rolling @ 1 to 200 days
                -Monthly-Rolling @ 1 to 11 months
                -Yearly-Rolling @ 1 to 10 years
    -roll_type: Daily Rolling or Monthly Rolling or Yearly Rolling 
    """

    # Check for Function Inputs Consistencies
    check_inputs = _check_input_roll_performance(data_df_timeline, roll_period, roll_type)

    # Rolling factor to scale data_df_timeline data to roll_type
    scaling_factor_roll_type = _scale_factor_roll_performance(data_df_timeline, roll_type)

    # Roll period
    roll_period = roll_period * scaling_factor_roll_type
    
    # Figure out annualization factor
    timeline_based_factor = _annualization_factor_roll_performance(data_df_timeline)
    annualization_factor = timeline_based_factor / roll_period
    
    # Check if Data size is less than roll period
    if data_df.shape[0] < roll_period:
        raise ValueError("Data Size is less than roll period. Not possible to run")

    # Compute Rolling Returns
    rolling_returns = data_df.add(1).rolling(roll_period).apply(np.prod).dropna().sub(1)
    ann_rolling_returns = rolling_returns.apply(lambda x: (1 + x) ** (annualization_factor)).sub(1)
    
    # Compute Stats
    summary_df = rolling_returns.describe()
    summary_df = summary_df.drop("std", axis = 0)
    
    
    # Annualize Rolling Returns Stats
    summary_df.iloc[1:,:] = summary_df.iloc[1:,:].apply(lambda x: (1 + x) ** (annualization_factor)).sub(1)
    
    # Compute Probability
    summary_df.loc["<0%P"] = _compute_probability_roll_performance(ann_rolling_returns, upper_bound = 0)
    summary_df.loc["0-10%P"] = _compute_probability_roll_performance(ann_rolling_returns, lower_bound = 0, upper_bound = 0.1)
    summary_df.loc["10-20%P"] = _compute_probability_roll_performance(ann_rolling_returns, lower_bound = 0.1, upper_bound = 0.2)
    summary_df.loc[">20%P"] = _compute_probability_roll_performance(ann_rolling_returns, lower_bound = 0.2)

    # Round it up and Get rid of leading zeros
    summary_df.iloc[1:, :] = summary_df.iloc[1:, :] * 100
    summary_df = np.round(summary_df, 2)

    # Voila/Hari-Om
    return summary_df.transpose()

In [18]:
# Blank for test

---

[Go To Index](#Index)
<a id="S5"></a>

## Calender Year Performance

In [19]:
def compute_calanderYear_performance(data_df, data_df_timeline = "daily"):
    """
    - Compute Calendar-Year performance
    - Drop any incomplete timelines from computation
        -- data_df_timeline is 'Daily': If a year has <240 days, we don't consider it for calendar year
        -- data_df_timeline is 'Monthly': If a year has <12 months, we don't consider it for calendar year
    """
    # Check Input
    data_df_timeline = data_df_timeline.strip().lower()
    if data_df_timeline not in ["daily", "monthly"]:
        raise ValueError("data_df_timeline input incorrect. Can only run for Daily or Monthly")

    # Scrape name of Date Col and Drop Date as Index
    date_col = data_df.index.name
    data = data_df.reset_index()

    # Convert to Year
    ## Check datatype: We are insuring for PeriodType, Object, and Datetime
    if isinstance(data[date_col].dtype, pd.core.dtypes.dtypes.PeriodDtype):
        data[date_col] = data[date_col].dt.to_timestamp().dt.to_period("Y")
    else:
        data[date_col] = pd.to_datetime(data[date_col], errors='coerce').dt.to_period("Y")
        
    # Figure out Relevant Calender Years
    cut_off_timeline = 230 if (data_df_timeline == "daily") else 11

    # Find out Relevant Years - that make the cutoff
    relevant_years = data[date_col].unique()[data[date_col].value_counts().sort_index() > cut_off_timeline]
    
    # Filter out data that makes the cutoff
    data_filter = data.loc[data[date_col].isin(relevant_years)]

    # Compute Calender Year Returns
    summary_df = data_filter.groupby(date_col).apply(lambda x: (1 + x).prod() - 1)

    # Round-up
    
    summary_df = summary_df * 100
    summary_df = np.round(summary_df, 2)
    
    # Voila/Hari Om
    return summary_df

In [20]:
# Blank for test

In [21]:
# Blank for test

---

[Go To Index](#Index)
<a id="S6"></a>

## Point in Time - Trailing Returns

In [22]:
def _lookback_returns_pit_R(data_df, lookback_period):
    """
    Compute periodic returns based on lookback_period
    """
    if data_df.shape[0] < lookback_period:
        return pd.Series(np.zeros(data_df.shape[1]), index = data_df.columns)
    else: return data_df.tail(lookback_period).add(1).prod().sub(1)

In [23]:
def _Daily_based_returns_pit_R(data_df, timeline_based_factor):
    """
    Compute Daily and Weekly returns
    """
    
    daily_df = pd.DataFrame(_lookback_returns_pit_R(data_df, int(timeline_based_factor / 252)), columns = ["1-d"])
    daily_df["1-w"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor / 50))
    return daily_df

In [24]:
def _D_Monthly_based_returns_pit_R(data_df, timeline_based_factor):
    """
    Compute 1-month, 3-months, and 6-months returns
    """
    
    d_Monthly_df = pd.DataFrame(_lookback_returns_pit_R(data_df, int(timeline_based_factor / 12)), columns = ["1-m"])
    d_Monthly_df["3-m"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor / 4))
    d_Monthly_df["6-m"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor / 2))
    return d_Monthly_df

In [25]:
def _DM_Yearly_based_returns_pit_R(data_df, timeline_based_factor):
    """
    Compute 1-year, 3-years, 5-years, and 10-years returns
    """
    
    _DM_Yearly_df = pd.DataFrame(_lookback_returns_pit_R(data_df, int(timeline_based_factor * 1)), columns = ["1-year"])
    
    _DM_Yearly_df["3-years"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor * 3)).add(1) ** (1 / 3)
    _DM_Yearly_df["3-years"] = _DM_Yearly_df["3-years"] - 1
    
    _DM_Yearly_df["5-years"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor * 5)).add(1) ** (1 / 5)
    _DM_Yearly_df["5-years"] = _DM_Yearly_df["5-years"] - 1
    
    _DM_Yearly_df["10-years"] = _lookback_returns_pit_R(data_df, int(timeline_based_factor * 10)).add(1) ** (1 / 10)
    _DM_Yearly_df["10-years"] = _DM_Yearly_df["10-years"] - 1
    
    return _DM_Yearly_df

In [26]:
def _check_timeline_pit_R(data_df, data_df_timeline):

    # Build Threshold
    days_threshold_dict = {"daily": 5, "monthly": 32, "yearly": 252}
    days_threshold = days_threshold_dict[data_df_timeline]

    # Bring Data DF in correct Format
    data = data_df.copy(deep = True)
    date_col = data.index.name
    data = data.reset_index()

    # Count days
    if isinstance(data[date_col].dtype, pd.core.dtypes.dtypes.PeriodDtype):
        data[date_col] = pd.to_datetime(data[date_col].dt.to_timestamp())
    else:
        data[date_col] = pd.to_datetime(data[date_col], errors='coerce')
    
    day_count = (data.iloc[-1, 0] - data.iloc[-2, 0]).days

    # Check for threshold
    if not (day_count > days_threshold - 5) & (day_count < days_threshold):
        raise ValueError(f"The data_df rows are not {data_df_timeline} format. Please re-check")
    else:
        return None

In [27]:
def compute_point_in_time_returns(data_df, data_df_timeline = "daily"):
    """
    -data_df: Quote a dataframe
    -data_df_timeline: The timeline of the dataframe quoted - daily or monthly or yearly
    """

    # Check Input
    data_df_timeline = data_df_timeline.strip().lower()
    if data_df_timeline not in ["daily", "monthly", "yearly"]:
        raise ValueError("data_df_timeline input incorrect. Can only run for Daily or Monthly")

    # Check data_df timeline format
    _check_timeline_pit_R(data_df, data_df_timeline)
    
    # Figure out the annualization factor of the data
    ## eg, if its daily, then the annualizaton is 252-days
    timeline_based_factor = _annualization_factor_roll_performance(data_df_timeline)


    # If daily compute - Daily, Weekly, Monthly, and Yearly
    if data_df_timeline == "daily":   
        summary_df = _Daily_based_returns_pit_R(data_df, timeline_based_factor)
        summary_df = (summary_df.merge
                      (_D_Monthly_based_returns_pit_R(data_df, timeline_based_factor),
                       left_index = True, right_index = True, how = "left")
                     )
        summary_df = (summary_df.merge
                      (_DM_Yearly_based_returns_pit_R(data_df, timeline_based_factor),
                       left_index = True, right_index = True, how = "left")
                     )
        summary_df = summary_df.drop(["1-d", "1-w"], axis = 1)
     # if its monthly, compute - Monthly, and Yearly
    elif data_df_timeline == "monthly":    
        summary_df = _D_Monthly_based_returns_pit_R(data_df, timeline_based_factor)
        summary_df = (summary_df.merge
                      (_DM_Yearly_based_returns_pit_R(data_df, timeline_based_factor),
                       left_index = True, right_index = True, how = "left")
                     )
        
     # if its yearly, compute - Yearly   
    else:
        summary_df = _DM_Yearly_based_returns_pit_R(data_df, timeline_based_factor)


    # Round-up
    summary_df = summary_df * 100
    summary_df = np.round(summary_df, 2)
    
    # Voila/Hari Om
    return summary_df

---

## No of Up-Down Months

In [28]:
def compute_no_up_down_months(monthly_returns_input):
    """
    -Compute no of up and down months for a returns series
    -Parameters:
    --monthly_returns_input: pd.DataFrame; Index: Dates; Columns: Returns Series
    """
    up_months = (monthly_returns_input > 0).sum()
    down_months = monthly_returns_input.shape[0] - (monthly_returns_input > 0).sum()

    up_down_df = pd.DataFrame((up_months.values, down_months.values), 
                              index = ["Up_Months", "Down_Months"],
                              columns = monthly_returns_input.columns
                             )

    return up_down_df

In [29]:
# Blank for test

In [30]:
# Blank for test

<hr style="border:4px solid black">

[Go To Index](#Index)
<a id="S7"></a>

# Run Code and Save on Excel

### Import File

In [31]:
# Input Folder
input_file_folder_name = "sample_input"

# Output Folder
output_file_folder_name = "comparison_output"

# File Name
custom_df_file_name = "sample_daily_returns.xlsx"
summary_stats_file_name = "notebook_output.xlsx"

# Build Path
custom_df_filepath = os.path.join(input_file_folder_name, custom_df_file_name)

# Repair File
custom_df = pd.read_excel(custom_df_filepath, sheet_name="Sheet1", index_col="Date")

# Sub-Chart Size for Bell-Curve
## There are two bell-charts we run. The first one will have all return series
## This is a parameter for the second-chart. How many of the return series do you need here?
sub_chart_size = 2

In [32]:
# Preview
custom_df

,QVM-VM,Stop-Loss,QVM-3m,VM-1m,QVM-3m_SL,Multifactor_PMS,NIFTY-500
Date,,,,,,,
2012-10-01,0.007951,-0.001658,0.006150,-0.008757,0.019183,0.008860,0.009171
2012-10-02,-0.001574,0.012170,-0.009359,-0.023384,0.007935,0.009088,0.009083
2012-10-03,0.010215,0.003309,-0.014576,0.011966,0.020662,0.009871,-0.025023
2012-10-04,0.023345,-0.017697,0.001818,0.014983,-0.028117,-0.003242,-0.001378
2012-10-05,-0.003012,-0.007050,0.005418,0.000710,0.006396,0.007675,0.015525
...,...,...,...,...,...,...,...
2025-10-27,-0.009624,-0.007965,0.022314,-0.005099,0.012914,-0.003215,-0.005919
2025-10-28,-0.005546,-0.020538,-0.008001,0.037587,0.000947,-0.002715,0.008167
2025-10-29,0.030373,0.000228,-0.005821,0.021282,0.012194,-0.008165,0.009653


---

### Designate Variables for the Code Run

In [33]:
# Declare - Variables
analysis_df = custom_df.copy(deep = True)

last_date_analysis = str(analysis_df.index[-1])[:10]
analysis_df_timeline = "daily"
analysis_roll_type = "yearly"

# Chart Legend
chart_legend_01 = "Oct-2012 to Oct-2025"

# Sheet Names
period_inception_name = "2012_2025"

# Sub Periods

sP_01_startY = 2013
sP_01_endY = 2015
sPeriod_01_name = "2013_2015"

sP_02_startY = 2016
sP_02_endY = 2020
sPeriod_02_name = "2016_2020"

sP_03_startY = 2021
sP_03_endY = 2025
sPeriod_03_name = "2021_2025"

# sP_04_startY = 2021
# sP_04_endY = 2025
# sPeriod_04_name = "2021_2025"


roll_01_name = "Rolling_1Y"
roll_03_name = "Rolling_3Y_ann"
roll_05_name = "Rolling_5Y_ann"
roll_10_name = "Rolling_10Y_ann"

calender_name = "Calender_Year"

pit_returns_name = f"Trailing_Returns_{last_date_analysis}"

# For excel output of the data series
final_all_returns_for_xl = analysis_df.copy(deep = True)
final_all_returns_for_xl.index = final_all_returns_for_xl.index.date
final_all_returns_for_xl.index.name = "Date"

---

### Run Analysis

In [34]:
# Run analysis - Call Functions
## Periodic Performance
periodic_inception = compute_portfolio_performance(analysis_df, data_df_timeline=analysis_df_timeline)
periodic_inception.index.name = period_inception_name

periodic_sb01 = compute_portfolio_performance(analysis_df, data_df_timeline=analysis_df_timeline, start_year=sP_01_startY, end_year=sP_01_endY)
periodic_sb01.index.name = sPeriod_01_name

periodic_sb02 = compute_portfolio_performance(analysis_df, data_df_timeline=analysis_df_timeline, start_year=sP_02_startY, end_year=sP_02_endY)
periodic_sb02.index.name = sPeriod_02_name

periodic_sb03 = compute_portfolio_performance(analysis_df, data_df_timeline=analysis_df_timeline, start_year=sP_03_startY, end_year=sP_03_endY)
periodic_sb03.index.name = sPeriod_03_name

# periodic_sb04 = compute_portfolio_performance(analysis_df, data_df_timeline=analysis_df_timeline, start_year=sP_04_startY, end_year=sP_04_endY)
# periodic_sb04.index.name = sPeriod_04_name


## Rolling
rolling_01 = compute_rolling_performance(analysis_df, data_df_timeline = analysis_df_timeline, roll_period = 1, roll_type = analysis_roll_type)
rolling_01.index.name = roll_01_name

rolling_03 = compute_rolling_performance(analysis_df, data_df_timeline = analysis_df_timeline, roll_period = 3, roll_type = analysis_roll_type)
rolling_03.index.name = roll_03_name

rolling_05 = compute_rolling_performance(analysis_df, data_df_timeline = analysis_df_timeline, roll_period = 5, roll_type = analysis_roll_type)
rolling_05.index.name = roll_05_name

rolling_10 = compute_rolling_performance(analysis_df, data_df_timeline = analysis_df_timeline, roll_period = 10, roll_type = analysis_roll_type)
rolling_10.index.name = roll_10_name


## Calender Year
calender_df = compute_calanderYear_performance(analysis_df, data_df_timeline = analysis_df_timeline)
calender_df.index.name = calender_name

## Point in Time Returns
pit_returns = compute_point_in_time_returns(analysis_df, data_df_timeline = analysis_df_timeline)
pit_returns.index.name = pit_returns_name

# Build monthly-returns if data not in monthly format
if analysis_df_timeline != "monthly":

    ## Build Month Tags for data
    for_monthly_df = analysis_df.copy(deep = True)
    for_monthly_df = for_monthly_df.reset_index()
    for_monthly_df["YM"] = for_monthly_df["Date"].dt.to_period("M")
    for_monthly_df = for_monthly_df.set_index("Date")
    
    ### Filter out months with less than 15 days
    mask = for_monthly_df["YM"].value_counts() > 15
    mask = mask[mask].index
    for_monthly_df= for_monthly_df.loc[for_monthly_df["YM"].isin(mask)]
    
    ### Compute monthly returns
    for_monthly_df = for_monthly_df.groupby("YM").apply(lambda x: (1+x).prod() - 1)
    for_monthly_df = for_monthly_df.to_timestamp("M")
    for_monthly_df.index = for_monthly_df.index.date
    for_monthly_df.index.name = "Date"

else:
    for_monthly_df = analysis_df.copy(deep = True)

In [35]:
# Compute No of Up-Down months
up_down_months_df = compute_no_up_down_months(for_monthly_df)
up_down_months_df.index.name = period_inception_name

In [36]:
# Regime - Crisis

# Step 1: Quote the Crisis-Regimes info
crisis_regimes = {
    "Global Financial Crisis": {
        "crisis_start": "08-01-2008",
        "crisis_end": "27-10-2008",
        "recovery_start": "28-10-2008",
        "recovery_end": "31-10-2009"
    },
    "Taper Tantrum": {
        "crisis_start": "01-01-2013",
        "crisis_end": "30-08-2013",
        "recovery_start": "31-08-2013",
        "recovery_end": "31-08-2014"
    },
    "Yuan Devaluation": {
        "crisis_start": "03-08-2015",
        "crisis_end": "29-02-2016",
        "recovery_start": "01-03-2016",
        "recovery_end": "28-02-2017"
    },
    "Covid Crash": {
        "crisis_start": "20-02-2020",
        "crisis_end": "31-03-2020",
        "recovery_start": "01-04-2020",
        "recovery_end": "31-12-2020"
    }
}

# Step 2: Loop over regime and compute returns
crisis_regimes_bucket = []
for _regime, _period in crisis_regimes.items():

    # Step 2.1: Define timeline
    cris_start = pd.to_datetime(_period["crisis_start"], format = "%d-%m-%Y")
    cris_end = pd.to_datetime(_period["crisis_end"], format = "%d-%m-%Y")
    recov_start = pd.to_datetime(_period["recovery_start"], format = "%d-%m-%Y")
    recov_end = pd.to_datetime(_period["recovery_end"], format = "%d-%m-%Y")

    # Step 2.2 Compute returns
    cris_returns = analysis_df.loc[cris_start:cris_end, :].add(1).prod().sub(1).values * 100
    recov_returns = analysis_df.loc[recov_start:recov_end, :].add(1).prod().sub(1).values * 100
    if (cris_returns == 0).all() and (recov_returns == 0).all():
        continue

    # Step 2.3: Build dataframe
    regime_ret_df = pd.DataFrame({_regime: cris_returns, f"Recovery {_regime}": recov_returns}, index=custom_df.columns).T
    regime_ret_df.insert(0, "Start_Date", [cris_start.strftime("%d-%m-%Y"), recov_start.strftime("%d-%m-%Y")])
    regime_ret_df.insert(1, "End_Date", [cris_end.strftime("%d-%m-%Y"), recov_end.strftime("%d-%m-%Y")])

    # Step 2.5: Store Returns-DF
    crisis_regimes_bucket.append(regime_ret_df)

# Step 3: Put together Crisis-Regime database
crisis_regimes_df = pd.concat(crisis_regimes_bucket)
crisis_regimes_df.index.name = "Crisis-Regimes"

In [37]:
# Regime - Markets

# Step 1: Quote the Market-Regimes info
market_regimes = {
    "Bear Regime 01": {
        "start": "01-01-2008",
        "end": "30-11-2008"
    },
    "Recovery Regime 01": {
        "start": "01-12-2008",
        "end": "31-10-2010"
    },
    "Bear Regime 02": {
        "start": "01-11-2010",
        "end": "31-12-2011"
    },
    "Recovery Regime 02": {
        "start": "01-01-2012",
        "end": "28-02-2014"
    },
    "Bull Regime 02": {
        "start": "01-03-2014",
        "end": "28-02-2015"
    },
    "Bear Regime 03": {
        "start": "01-03-2015",
        "end": "29-02-2016"
    },
    "Recovery Regime 03": {
        "start": "01-03-2016",
        "end": "31-12-2016"
    },
    "Bull Regime 03": {
        "start": "01-01-2017",
        "end": "31-01-2020"
    },
    "Bear Regime 04": {
        "start": "01-02-2020",
        "end": "31-03-2020"
    },
    "Recovery Regime 04": {
        "start": "01-04-2020",
        "end": "31-10-2020"
    },
    "Bull Regime 04": {
        "start": "01-11-2020",
        "end": "28-02-2022"
    },
    "Bear Regime 05": {
        "start": "01-03-2022",
        "end": "31-07-2022"
    },
    "Bull Regime 05": {
        "start": "01-08-2022",
        "end": "30-09-2024"
    },
    "Bear Regime 06": {
        "start": "01-10-2024",
        "end": "28-02-2025"
    }
}

# Step 2: Loop over regime and compute returns
market_regimes_bucket = []
for _regime, _period in market_regimes.items():

    # Step 2.1: Define timeline
    reg_start = pd.to_datetime(_period["start"], format = "%d-%m-%Y")
    reg_end = pd.to_datetime(_period["end"], format = "%d-%m-%Y")

    # Step 2.2 Compute returns
    reg_returns = analysis_df.loc[reg_start:reg_end, :].add(1).prod().sub(1).values * 100
    if (reg_returns == 0).all():
        continue

    # Step 2.3: Build dataframe
    mark_regime_ret_df = pd.DataFrame({_regime: reg_returns}, index = custom_df.columns).T
    mark_regime_ret_df.insert(0, "Start_Date", reg_start.strftime("%d-%m-%Y"))
    mark_regime_ret_df.insert(1, "End_Date", reg_end.strftime("%d-%m-%Y"))

    # Step 2.5: Store Returns-DF
    market_regimes_bucket.append(mark_regime_ret_df)

# Step 3: Put together Crisis-Regime database
market_regimes_df = pd.concat(market_regimes_bucket)
market_regimes_df.index.name = "Market-Regimes"

---

### Build Plots

In [38]:
# Build Growth of Wealth Chart
gofwealth = for_monthly_df.add(1).cumprod().multiply(10000).reset_index()
last_date = gofwealth["Date"].iloc[-1].strftime("%Y-%m-%d")

plt.figure(figsize=(12, 6))
colors = plt.colormaps["tab10"].colors


for idx, col in enumerate(gofwealth.columns[1:]):
    plt.plot(gofwealth["Date"], gofwealth[col], label=col, linewidth=1, color=colors[idx % len(colors)])


plt.ylabel("Growth of Rs 10,000", fontsize = 10)
plt.title(f"Growth of Rs 10,000 {chart_legend_01}", fontsize = 12)
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('Rs. {x:,.0f}'))
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)
plt.legend(fontsize=8)
plt.tight_layout()

# Save the plot to a BytesIO buffer
gow_chart = io.BytesIO()
plt.savefig(gow_chart, format="png", dpi = 300)
plt.close()  # Close the plot to avoid display
gow_chart.seek(0)

0

In [39]:
# Build Drawdown Chart

drawdown_df = (np.log(1 + for_monthly_df).cumsum()) - (np.log(1 + for_monthly_df).cumsum().cummax()) 
drawdown_df = (np.exp(drawdown_df) - 1)
drawdown_df = drawdown_df.reset_index()

# Drawdown Chart
plt.figure(figsize=(12, 6))
colors = plt.colormaps["tab10"].colors  # Same as Growth of Wealth

for idx, col in enumerate(drawdown_df.columns[1:]):
    plt.plot(drawdown_df["Date"], drawdown_df[col] * 100, label=col, linewidth=0.75, color=colors[idx % len(colors)])

plt.axhline(0, color='black', linewidth=0.5, linestyle='--')  # Zero line for reference

plt.ylabel("Drawdown (%)", fontsize=10)
plt.title(f"Monthly Drawdowns {chart_legend_01}", fontsize=12)
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}%'))  # Percentage format

plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.legend(fontsize=8)
plt.tight_layout()

# Save to buffer
drawdown_chart = io.BytesIO()
plt.savefig(drawdown_chart, format="png", dpi=300)
plt.close()
drawdown_chart.seek(0)

0

In [40]:
# Build CY Heatmap

# Calculate size based on data shape
n_rows, n_cols = calender_df.shape
cell_width = 0.8
cell_height = 0.3  

fig_width = max(6, n_cols * cell_width)
fig_height = max(4, n_rows * cell_height)

heatmap_data = calender_df
plt.figure(figsize=(fig_width, fig_height))

# Create heatmap
ax = sns.heatmap(heatmap_data,
                 annot=True,
                 fmt=".1f",
                 cmap="RdYlGn",
                 center=0,
                 linewidths=0.5,
                 cbar_kws={"label": "Return (%)"})

# Formatting
plt.title("Calender Year Returns Heatmap", fontsize=12)
ax.set_xlabel("Strategies") 
ax.set_ylabel("Calender Year", fontsize=10)
plt.xticks(rotation=45, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

# Save to buffer
cy_heatmap = io.BytesIO()
plt.savefig(cy_heatmap, format="png", dpi=300)
plt.close()
cy_heatmap.seek(0)

0

In [41]:
# Build Correlation Heatmap
# Compute correlation matrix
corr_matrix = analysis_df.corr()

# Plot
plt.figure(figsize=(6, 4))
ax = sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)

plt.title("Correlation Matrix Heatmap", fontsize=12)
ax.set_xlabel("Strategies-Since Inception") 
ax.set_ylabel("Strategies-Since Inception", fontsize=10)

plt.xticks(rotation=45, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

# Save to buffer
corr_heatmap = io.BytesIO()
plt.savefig(corr_heatmap, format="png", dpi=300)
plt.close()
corr_heatmap.seek(0)

0

In [42]:
# Build Normal Distribution
# Melt the DataFrame
long_df = for_monthly_df.melt(var_name = "Strategies", value_name = "Monthly Return")

# Set the plot size
plt.figure(figsize=(12, 6))


# Set up color mapping using tab10
colors = plt.colormaps["tab10"].colors
assets = for_monthly_df.columns.tolist()
palette = {asset: colors[i % len(colors)] for i, asset in enumerate(assets)}

# Plot histogram
sns.kdeplot(
    data=long_df,
    x='Monthly Return',
    hue='Strategies',
    palette=palette,
    common_norm=False,
    linewidth=2
)

plt.title('Distribution of Monthly Returns')
plt.xlabel('Monthly Return')
plt.ylabel('Density')
plt.grid(True)
plt.tight_layout()


# Save to buffer
bell_curve = io.BytesIO()
plt.savefig(bell_curve, format="png", dpi=300)
plt.close()
bell_curve.seek(0)

0

In [43]:
# Sub-Set Normal-Dist/Bell-Curve
# Add Normal-Dist for sub-set if it exists:
if for_monthly_df.shape[1] > sub_chart_size:
    filtered_monthly_df = for_monthly_df.copy(deep = True)
    filtered_monthly_df = filtered_monthly_df.iloc[:, :sub_chart_size]
    
    # Melt the DataFrame
    long_df = filtered_monthly_df.melt(var_name = "Strategies", value_name = "Monthly Return")
    
    # Set the plot size
    plt.figure(figsize=(12, 6))
    
    
    # Set up color mapping using tab10
    colors = plt.colormaps["tab10"].colors
    assets = for_monthly_df.columns.tolist()
    palette = {asset: colors[i % len(colors)] for i, asset in enumerate(assets)}
    
    # Plot histogram
    sns.kdeplot(
        data=long_df,
        x='Monthly Return',
        hue='Strategies',
        palette=palette,
        common_norm=False,
        linewidth=2
    )
    
    plt.title('Distribution of Monthly Returns')
    plt.xlabel('Monthly Return')
    plt.ylabel('Density')
    plt.grid(True)
    plt.tight_layout()
    
    
    # Save to buffer
    bell_curve_subset = io.BytesIO()
    plt.savefig(bell_curve_subset, format="png", dpi=300)
    plt.close()
    bell_curve_subset.seek(0)

In [44]:
# Build Box-Plot
plt.figure(figsize=(12, 6))

# Set up colors using tab10
colors = plt.colormaps["tab10"].colors
assets = for_monthly_df.columns.tolist()
palette = {asset: colors[i % len(colors)] for i, asset in enumerate(assets)}

# Create boxplot
sns.boxplot(data = for_monthly_df, legend = True, palette = palette)

plt.title('Box-Whisker Plot of Monthly Returns')
plt.xlabel('Asset')
plt.ylabel('Monthly Return')

# Manually create legend and place it outside the plot
handles = [plt.Line2D([], [], marker='s', linestyle='None', color=palette[asset], label=asset)
           for asset in assets]
plt.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

plt.tight_layout()

# Save to buffer
box_plot = io.BytesIO()
plt.savefig(box_plot, format="png", dpi=300)
plt.close()
box_plot.seek(0)

0

---

### Save File

In [45]:
# Save File

## Create File and Path
output_folder_path = os.path.join(output_file_folder_name, summary_stats_file_name)

## Write File
with pd.ExcelWriter(output_folder_path) as writer:
    
    # Input Data
    final_all_returns_for_xl.reset_index().to_excel(writer, sheet_name = "input_returns", index = False)

    # Periodic Data
    ## Since Inception
    start_row = 0
    periodic_inception.reset_index().to_excel(writer, startrow = start_row, sheet_name = "periodic_returns", index = False)
    
    
    ## Sub-Period 01
    start_row = periodic_inception.shape[0] + 5
    periodic_sb01.reset_index().to_excel(writer, startrow = start_row, sheet_name = "periodic_returns", index = False)

    ## Sub-Period 02
    start_row += periodic_sb01.shape[0] + 5
    periodic_sb02.reset_index().to_excel(writer, startrow = start_row, sheet_name = "periodic_returns", index = False)

    ## Sub-Period 03
    start_row += periodic_sb02.shape[0] + 5
    periodic_sb03.reset_index().to_excel(writer, startrow = start_row, sheet_name = "periodic_returns", index = False)

    ## Sub-Period 04
    # start_row += periodic_sb03.shape[0] + 5
    # periodic_sb04.reset_index().to_excel(writer, startrow = start_row, sheet_name = "periodic_returns", index = False)


    # Rolling Data
    ## Rolling 1-Year
    start_row = 0
    rolling_01.reset_index().to_excel(writer, startrow = start_row, sheet_name = "rolling_returns", index = False)

    ## Rolling 3-Year
    start_row = rolling_01.shape[0] + 5
    rolling_03.reset_index().to_excel(writer, startrow = start_row, sheet_name = "rolling_returns", index = False)

    ## Rolling 5-Year
    start_row += rolling_03.shape[0] + 5
    rolling_05.reset_index().to_excel(writer, startrow = start_row, sheet_name = "rolling_returns", index = False)

    ## Rolling 10-Year
    start_row += rolling_05.shape[0] + 5
    rolling_10.reset_index().to_excel(writer, startrow = start_row, sheet_name = "rolling_returns", index = False)


    # Calender Year
    calender_df.reset_index().to_excel(writer, sheet_name = "calender_returns", index = False)
    # Insert heatmap into that same sheet
    writer.sheets["calender_returns"].insert_image('J2', "cy_heatmap.png", {"image_data": cy_heatmap})

    # Monthly Returns
    for_monthly_df.reset_index().to_excel(writer, sheet_name = "monthly_returns", index = False)

    # No of Up and Down months
    up_down_months_df.reset_index().to_excel(writer, sheet_name = "no_of_up-down_months", index = False)
    
    # Trailing Returns
    pit_returns.reset_index().to_excel(writer, sheet_name = "trailing_returns", index = False)

    # Crisis-Regimes
    crisis_regimes_df.reset_index().to_excel(writer, sheet_name = "crisis_regimes", index = False)

    # Market-Regimes
    market_regimes_df.reset_index().to_excel(writer, sheet_name = "market_regimes", index = False)

    # Add chart
    if "charts_01" not in writer.sheets:
        writer.sheets["charts_01"] = writer.book.add_worksheet("charts_01")
    writer.sheets["charts_01"].insert_image('A2', "plot.png", {"image_data": bell_curve})
    writer.sheets["charts_01"].insert_image('A35', "plot.png", {"image_data": box_plot})

    # Sub-Set Normal-Dist/Bell-Curve
    # Add Normal-Dist for sub-set if it exists:
    if for_monthly_df.shape[1] > sub_chart_size:
        writer.sheets["charts_01"].insert_image('O2', "plot.png", {"image_data": bell_curve_subset})
    
    if "charts_02" not in writer.sheets:
        writer.sheets["charts_02"] = writer.book.add_worksheet("charts_02")
    writer.sheets["charts_02"].insert_image('A2', "plot.png", {"image_data": drawdown_chart})
    writer.sheets["charts_02"].insert_image('A35', "plot.png", {"image_data": gow_chart})
    writer.sheets["charts_02"].insert_image('N2', "plot.png", {"image_data": corr_heatmap})

    
    # Set the column width and number format
    workbook = writer.book
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        
        # Create a format for numbers with 2 decimal places
        format_decimal = workbook.add_format({'num_format': '0.00'})
        percent_format = writer.book.add_format({'num_format': '0.0%', 'align': 'right'})
        date_format = writer.book.add_format({'num_format': 'yyyy-mm-dd'})

        # Customize sheet format
        if sheet_name == "input_returns" or sheet_name == "monthly_returns":
            worksheet.set_column('A:A', 20, date_format) 
            worksheet.set_column('B:Z', 15, percent_format)
        elif sheet_name == "calender_returns":
            worksheet.set_column('A:Z', 15, format_decimal)
        elif sheet_name == "trailing_returns":
            worksheet.set_column('A:A', 25) 
            worksheet.set_column('B:Z', 12, format_decimal)
        elif sheet_name == "crisis_regimes" or sheet_name == "market_regimes":
            worksheet.set_column('A:A', 25)
            worksheet.set_column('B:C', 20, date_format)
            worksheet.set_column('D:Z', 12, format_decimal)
            
        else:
            # Apply width and format to all numerical columns (e.g., A to Z)
            worksheet.set_column('A:A', 20,) 
            worksheet.set_column('B:Z', 12, format_decimal) 

<hr style="border:4px solid black">

[Go To Index](#Index)
<a id="S9"></a>